In [0]:
#!/usr/bin/env python3
"""

特点：
- 纯函数调用，不依赖 argparse
- 纯 kafka-python
- 支持：列出消费者组、查看消费情况、修改 offset（dry-run / execute）
"""

from __future__ import annotations

from datetime import datetime
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

from kafka import KafkaAdminClient, KafkaConsumer, TopicPartition
from kafka.structs import OffsetAndMetadata

import json


@dataclass
class KafkaNotebookConfig:
    bootstrap_server: str
    client_kwargs: Dict[str, Any]


def parse_time_to_timestamp_ms(time_input: Any) -> int:
    """
    将时间输入解析为毫秒时间戳。

    支持：
    - int/float：直接视为毫秒时间戳
    - Local_Date_Time 字符串：
      - 2026-06-02T10:30:00
      - 2026-06-02 10:30:00
      - 2026-06-02T10:30:00.123
      - 2026-06-02T10:30:00Z
      - 2026-06-02T10:30:00+08:00

    说明：
    - 若字符串不带时区（LocalDateTime），按当前运行环境的本地时区解释。
    - 本函数输出的是 epoch 毫秒（绝对时间点），后续传给 Kafka 查询时不再涉及时区歧义。
    """
    if isinstance(time_input, (int, float)):
        ts = int(time_input)
        if ts < 0:
            raise ValueError("时间戳必须是非负毫秒数")
        return ts

    if not isinstance(time_input, str):
        raise ValueError("time_input 必须是毫秒时间戳(int/float)或时间字符串(str)")

    raw = time_input.strip()
    if not raw:
        raise ValueError("time_input 不能为空")

    # 兼容 ISO8601 的 Z 后缀（UTC），统一替换为 fromisoformat 可解析的 +00:00。
    norm = raw.replace("Z", "+00:00")

    dt: Optional[datetime] = None
    try:
        dt = datetime.fromisoformat(norm)
    except ValueError:
        # 兜底常见格式。
        fmts = [
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d %H:%M:%S.%f",
            "%Y-%m-%dT%H:%M:%S",
            "%Y-%m-%dT%H:%M:%S.%f",
        ]
        for fmt in fmts:
            try:
                dt = datetime.strptime(raw, fmt)
                break
            except ValueError:
                continue

    if dt is None:
        raise ValueError(
            "无法解析时间，请使用毫秒时间戳或 Local_Date_Time 格式，例如 2026-06-02T10:30:00"
        )

    # 对无时区时间按当前环境本地时区解释。
    # 例如 Databricks 集群若为 UTC，则 2026-06-02T10:30:00 会按 UTC 10:30 解析。
    if dt.tzinfo is None:
        dt = dt.astimezone()

    return int(dt.timestamp() * 1000)


def _commit_offsets_via_consumer_with_retry(
    config: KafkaNotebookConfig,
    group_id: str,
    topics: List[str],
    commit_map: Dict[TopicPartition, OffsetAndMetadata],
    retries: int = 3,
    join_timeout_ms: int = 5000,
) -> None:
    """
    兜底方案：通过 consumer commit 修改 offset。

    关键点：
    - 先 subscribe + poll 触发入组，再提交 offset，降低 UnknownMemberIdError 概率。
    - 遇到 UnknownMemberIdError 时重建 consumer 并重试。
    - 该路径仅作为 Admin API 不可用或异常时的兜底路径。
    """
    last_exc: Optional[Exception] = None
    for _ in range(retries):
        consumer = KafkaConsumer(
            bootstrap_servers=config.bootstrap_server,
            group_id=group_id,
            enable_auto_commit=False,
            **config.client_kwargs,
        )
        try:
            # 通过 poll 触发 group join，让 broker 分配有效 member 身份后再 commit。
            consumer.subscribe(topics)
            consumer.poll(timeout_ms=join_timeout_ms)
            consumer.commit(offsets=commit_map)
            return
        except Exception as exc:
            last_exc = exc
            # 非 UnknownMemberIdError 直接抛出，避免掩盖 ACL/认证/网络等真实问题。
            if "UnknownMemberIdError" not in str(exc):
                raise
        finally:
            consumer.close()

    if last_exc is not None:
        raise last_exc



def build_config(
    bootstrap_server: str,
    security_protocol: Optional[str] = None,
    sasl_mechanism: Optional[str] = None,
    sasl_plain_username: Optional[str] = None,
    sasl_plain_password: Optional[str] = None,
    ssl_cafile: Optional[str] = None,
    request_timeout_ms: int = 30000,
    extra_client_kwargs: Optional[Dict[str, Any]] = None,
) -> KafkaNotebookConfig:
    """构建 Kafka 配置，供 notebook 里复用。"""
    kwargs: Dict[str, Any] = {
        "request_timeout_ms": request_timeout_ms,
    }

    if security_protocol:
        kwargs["security_protocol"] = security_protocol
    if sasl_mechanism:
        kwargs["sasl_mechanism"] = sasl_mechanism
    if sasl_plain_username:
        kwargs["sasl_plain_username"] = sasl_plain_username
    if sasl_plain_password:
        kwargs["sasl_plain_password"] = sasl_plain_password
    if ssl_cafile:
        kwargs["ssl_cafile"] = ssl_cafile

    if extra_client_kwargs:
        kwargs.update(extra_client_kwargs)

    return KafkaNotebookConfig(
        bootstrap_server=bootstrap_server,
        client_kwargs=kwargs,
    )


def list_consumer_groups(config: KafkaNotebookConfig) -> List[Dict[str, str]]:
    """查看所有消费者组。"""
    admin = KafkaAdminClient(
        bootstrap_servers=config.bootstrap_server,
        **config.client_kwargs,
    )
    try:
        groups = admin.list_consumer_groups()
        return [
            {
                "group_id": group_id,
                "protocol_type": protocol_type,
            }
            for group_id, protocol_type in groups
        ]
    finally:
        admin.close()


def describe_consumer_group(
    config: KafkaNotebookConfig,
    group_id: str,
) -> List[Dict[str, int]]:
    """查看某个消费者组消费情况，返回 committed/end/lag。"""
    admin = KafkaAdminClient(
        bootstrap_servers=config.bootstrap_server,
        **config.client_kwargs,
    )
    consumer = KafkaConsumer(
        bootstrap_servers=config.bootstrap_server,
        enable_auto_commit=False,
        **config.client_kwargs,
    )

    try:
        committed = admin.list_consumer_group_offsets(group_id)
        if not committed:
            return []

        partitions = sorted(committed.keys(), key=lambda tp: (tp.topic, tp.partition))
        end_offsets = consumer.end_offsets(partitions)

        rows: List[Dict[str, int]] = []
        for tp in partitions:
            meta = committed.get(tp)
            committed_offset = meta.offset if meta is not None else -1
            log_end_offset = end_offsets.get(tp, -1)
            lag = (
                log_end_offset - committed_offset
                if committed_offset >= 0 and log_end_offset >= 0
                else -1
            )
            rows.append(
                {
                    "topic": tp.topic,
                    "partition": tp.partition,
                    "committed_offset": committed_offset,
                    "log_end_offset": log_end_offset,
                    "lag": lag,
                }
            )
        return rows
    finally:
        consumer.close()
        admin.close()


def set_consumer_group_offsets(
    config: KafkaNotebookConfig,
    group_id: str,
    topic: str,
    partition_offsets: Dict[int, int],
    execute: bool = False,
) -> Dict[str, Any]:
    """
    修改消费者组 offset。

    参数：
    - partition_offsets 示例：{0: 100, 1: 200}
    - execute=False 时仅 dry-run

    实现策略：
    - 首选 Admin API 直接修改消费组 offset（不依赖 member 身份，稳定性更好）。
    - 若环境不支持或触发 UnknownMemberIdError，则回退到 consumer commit 兜底。
    """
    admin = KafkaAdminClient(
        bootstrap_servers=config.bootstrap_server,
        **config.client_kwargs,
    )
    try:
        target_tps = [
            TopicPartition(topic=topic, partition=p)
            for p in sorted(partition_offsets.keys())
        ]
        existing = admin.list_consumer_group_offsets(group_id)

        # plan 用于在 dry-run/execute 两种模式下统一展示“当前值 -> 目标值”。
        plan = []
        for tp in target_tps:
            current_meta = existing.get(tp)
            current_offset = current_meta.offset if current_meta else -1
            plan.append(
                {
                    "topic": tp.topic,
                    "partition": tp.partition,
                    "current_offset": current_offset,
                    "target_offset": partition_offsets[tp.partition],
                }
            )

        # dry-run：只返回变更计划，不写 Kafka。
        if not execute:
            return {
                "mode": "dry-run",
                "plan": plan,
            }

        commit_map = {
            tp: OffsetAndMetadata(
                offset=partition_offsets[tp.partition],
                metadata="set-by-kafka-group-operator-notebook",
                leader_epoch=-1,
            )
            for tp in target_tps
        }

        applied_strategy = "admin.alter_consumer_group_offsets"

        # 优先使用 Admin API 修改消费组 offset，避免依赖 member 状态导致 UnknownMemberIdError。
        if hasattr(admin, "alter_consumer_group_offsets"):
            try:
                admin.alter_consumer_group_offsets(group_id, commit_map)
            except Exception as exc:
                # 部分版本/环境可能仍失败，回退到入组后提交并重试。
                if "UnknownMemberIdError" not in str(exc):
                    raise
                _commit_offsets_via_consumer_with_retry(
                    config=config,
                    group_id=group_id,
                    topics=[topic],
                    commit_map=commit_map,
                )
                applied_strategy = "consumer.commit(retry-after-join)"
        else:
            # 老版本 kafka-python 无该接口时回退到入组后提交并重试。
            _commit_offsets_via_consumer_with_retry(
                config=config,
                group_id=group_id,
                topics=[topic],
                commit_map=commit_map,
            )
            applied_strategy = "consumer.commit(retry-after-join)"

        # 提交后回读，返回真实落库值用于审计与校验。
        latest = admin.list_consumer_group_offsets(group_id)
        after = []
        for tp in target_tps:
            latest_meta = latest.get(tp)
            after.append(
                {
                    "topic": tp.topic,
                    "partition": tp.partition,
                    "offset_after_commit": latest_meta.offset if latest_meta else -1,
                }
            )

        return {
            "mode": "execute",
            "strategy": applied_strategy,
            "plan": plan,
            "after": after,
        }
    finally:
        admin.close()


def set_group_offsets_to_timestamp_prev(
    config: KafkaNotebookConfig,
    group_id: str,
    timestamp_ms: Any,
    execute: bool = False,
) -> Dict[str, Any]:
    """
    将指定消费者组下所有已提交分区的 offset 调整到“指定时间戳之前一条”。

        说明：
        - 作用范围：该消费组当前已提交过 offset 的全部 topic/partition。
        - 目标值计算：
            1) 若存在 timestamp_ms 对应位置 offset_x，则目标为 max(offset_x - 1, 0)
            2) 若不存在（通常是时间戳晚于分区最后一条消息时间），目标为 max(end_offset - 1, 0)
        - execute=False 时仅返回计划，不会真正提交。

        设计意图：
        - 该方法面向“按时间点整体回拨”场景，避免逐 topic/partition 手工计算 offset。
        - 返回 plan/after 两段数据，便于先预演、再执行、最后核对。
    """
        # 统一解析时间输入，支持毫秒时间戳或 Local_Date_Time 字符串。
    parsed_timestamp_ms = parse_time_to_timestamp_ms(timestamp_ms)

    admin = KafkaAdminClient(
        bootstrap_servers=config.bootstrap_server,
        **config.client_kwargs,
    )
    consumer = KafkaConsumer(
        bootstrap_servers=config.bootstrap_server,
        enable_auto_commit=False,
        **config.client_kwargs,
    )

    try:
        existing = admin.list_consumer_group_offsets(group_id)
        # 该组如果没有任何已提交 offset，说明没有可调整的分区。
        if not existing:
            return {
                "mode": "dry-run" if not execute else "execute",
                "group_id": group_id,
                "timestamp_ms": parsed_timestamp_ms,
                "plan": [],
                "message": "该消费者组没有已提交 offset，未执行任何修改。",
            }

        # 仅处理“该组已有提交记录”的分区，避免误操作无关分区。
        target_tps = sorted(existing.keys(), key=lambda tp: (tp.topic, tp.partition))
        end_offsets = consumer.end_offsets(target_tps)
        offsets_at_ts = consumer.offsets_for_times({tp: parsed_timestamp_ms for tp in target_tps})

        plan = []
        commit_map: Dict[TopicPartition, OffsetAndMetadata] = {}
        for tp in target_tps:
            current_meta = existing.get(tp)
            current_offset = current_meta.offset if current_meta is not None else -1

            oat = offsets_at_ts.get(tp)
            end_offset = end_offsets.get(tp, 0)
            if oat is not None:
                # 找到“>= 指定时间戳”的第一条消息 offset，向前回拨 1 条。
                target_offset = max(oat.offset - 1, 0)
                calc_reason = "from_offsets_for_times_minus_1"
            else:
                # 如果该时间点之后无数据（或空分区），回拨到分区末尾前一条。
                target_offset = max(end_offset - 1, 0)
                calc_reason = "timestamp_after_last_or_empty_partition"

            commit_map[tp] = OffsetAndMetadata(
                offset=target_offset,
                metadata="set-by-group-timestamp-prev",
                leader_epoch=-1,
            )
            plan.append(
                {
                    "topic": tp.topic,
                    "partition": tp.partition,
                    "current_offset": current_offset,
                    "target_offset": target_offset,
                    "end_offset": end_offset,
                    "calc_reason": calc_reason,
                }
            )

        # dry-run：只输出计划，不落库。
        if not execute:
            return {
                "mode": "dry-run",
                "group_id": group_id,
                "timestamp_ms": parsed_timestamp_ms,
                "plan": plan,
            }

        applied_strategy = "admin.alter_consumer_group_offsets"
        if hasattr(admin, "alter_consumer_group_offsets"):
            try:
                admin.alter_consumer_group_offsets(group_id, commit_map)
            except Exception as exc:
                # Admin API 遇到 member 相关异常时，回退到 consumer commit 重试。
                if "UnknownMemberIdError" not in str(exc):
                    raise
                _commit_offsets_via_consumer_with_retry(
                    config=config,
                    group_id=group_id,
                    topics=sorted(list({tp.topic for tp in target_tps})),
                    commit_map=commit_map,
                )
                applied_strategy = "consumer.commit(retry-after-join)"
        else:
            # 老版本 kafka-python 无 Admin API 时的兜底策略。
            _commit_offsets_via_consumer_with_retry(
                config=config,
                group_id=group_id,
                topics=sorted(list({tp.topic for tp in target_tps})),
                commit_map=commit_map,
            )
            applied_strategy = "consumer.commit(retry-after-join)"

        # 提交后回读每个分区的真实值，确保结果可核验。
        latest = admin.list_consumer_group_offsets(group_id)
        after = []
        for tp in target_tps:
            latest_meta = latest.get(tp)
            after.append(
                {
                    "topic": tp.topic,
                    "partition": tp.partition,
                    "offset_after_commit": latest_meta.offset if latest_meta else -1,
                }
            )

        return {
            "mode": "execute",
            "group_id": group_id,
            "timestamp_ms": parsed_timestamp_ms,
            "strategy": applied_strategy,
            "plan": plan,
            "after": after,
        }
    finally:
        consumer.close()
        admin.close()


def print_rows(rows: List[Dict[str, Any]]) -> None:
    """轻量表格输出，方便 notebook 快速查看。"""
    if not rows:
        print("(empty)")
        return

    headers = list(rows[0].keys())
    col_widths = {h: len(h) for h in headers}

    for row in rows:
        for h in headers:
            col_widths[h] = max(col_widths[h], len(str(row.get(h, ""))))

    header_line = " | ".join(h.ljust(col_widths[h]) for h in headers)
    sep_line = "-+-".join("-" * col_widths[h] for h in headers)
    print(header_line)
    print(sep_line)

    for row in rows:
        print(" | ".join(str(row.get(h, "")).ljust(col_widths[h]) for h in headers))

def to_json(value: str) -> str:
    import json
    return json.dumps(result)



In [0]:
# # Databricks notebook source
# # DBTITLE 1,func
# #!/usr/bin/env python3
# """
# 特点：
# - 纯函数调用，不依赖 argparse
# - 纯 confluent-kafka
# - 支持：列出消费者组、查看消费情况、修改 offset（dry-run / execute）
# """

# from __future__ import annotations

# from datetime import datetime
# from dataclasses import dataclass
# from typing import Any, Dict, List, Optional

# from confluent_kafka import Consumer, TopicPartition, KafkaException
# from confluent_kafka.admin import AdminClient, ConsumerGroupTopicPartitions


# @dataclass
# class KafkaNotebookConfig:
#     bootstrap_server: str
#     client_kwargs: Dict[str, Any]


# def parse_time_to_timestamp_ms(time_input: Any) -> int:
#     """
#     将时间输入解析为毫秒时间戳。

#     支持：
#     - int/float：直接视为毫秒时间戳
#     - Local_Date_Time 字符串：
#       - 2026-06-02T10:30:00
#       - 2026-06-02 10:30:00
#       - 2026-06-02T10:30:00.123
#       - 2026-06-02T10:30:00Z
#       - 2026-06-02T10:30:00+08:00

#     说明：
#     - 若字符串不带时区（LocalDateTime），按当前运行环境的本地时区解释。
#     - 本函数输出的是 epoch 毫秒（绝对时间点），后续传给 Kafka 查询时不再涉及时区歧义。
#     """
#     if isinstance(time_input, (int, float)):
#         ts = int(time_input)
#         if ts < 0:
#             raise ValueError("时间戳必须是非负毫秒数")
#         return ts

#     if not isinstance(time_input, str):
#         raise ValueError("time_input 必须是毫秒时间戳(int/float)或时间字符串(str)")

#     raw = time_input.strip()
#     if not raw:
#         raise ValueError("time_input 不能为空")

#     norm = raw.replace("Z", "+00:00")

#     dt: Optional[datetime] = None
#     try:
#         dt = datetime.fromisoformat(norm)
#     except ValueError:
#         fmts = [
#             "%Y-%m-%d %H:%M:%S",
#             "%Y-%m-%d %H:%M:%S.%f",
#             "%Y-%m-%dT%H:%M:%S",
#             "%Y-%m-%dT%H:%M:%S.%f",
#         ]
#         for fmt in fmts:
#             try:
#                 dt = datetime.strptime(raw, fmt)
#                 break
#             except ValueError:
#                 continue

#     if dt is None:
#         raise ValueError(
#             "无法解析时间，请使用毫秒时间戳或 Local_Date_Time 格式，例如 2026-06-02T10:30:00"
#         )

#     if dt.tzinfo is None:
#         dt = dt.astimezone()

#     return int(dt.timestamp() * 1000)


# def _admin_conf(config: KafkaNotebookConfig) -> Dict[str, Any]:
#     """组装 AdminClient 配置（librdkafka 风格 dict）。"""
#     cfg: Dict[str, Any] = {"bootstrap.servers": config.bootstrap_server}
#     cfg.update(config.client_kwargs)
#     return cfg


# def _consumer_conf(
#     config: KafkaNotebookConfig,
#     group_id: str,
#     enable_auto_commit: bool = False,
# ) -> Dict[str, Any]:
#     """组装 Consumer 配置（librdkafka 风格 dict）。"""
#     cfg: Dict[str, Any] = {
#         "bootstrap.servers": config.bootstrap_server,
#         "group.id": group_id,
#         "enable.auto.commit": enable_auto_commit,
#     }
#     cfg.update(config.client_kwargs)
#     return cfg


# def _list_group_offsets(admin: AdminClient, group_id: str) -> List[TopicPartition]:
#     """统一封装：返回该 group 的所有 committed TopicPartition（含 offset 字段）。"""
#     futures = admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(group_id)])
#     result = futures[group_id].result()
#     return list(result.topic_partitions or [])


# def _commit_offsets_via_consumer_with_retry(
#     config: KafkaNotebookConfig,
#     group_id: str,
#     topics: List[str],
#     commit_tps: List[TopicPartition],
#     retries: int = 3,
#     join_timeout_ms: int = 5000,
# ) -> None:
#     """
#     兜底方案：通过 consumer commit 修改 offset。

#     关键点：
#     - 先 subscribe + poll 触发入组，再提交 offset，降低 UnknownMemberIdError 概率。
#     - 遇到 UnknownMemberIdError 时重建 consumer 并重试。
#     - 该路径仅作为 Admin API 不可用或异常时的兜底路径。
#     """
#     last_exc: Optional[Exception] = None
#     for _ in range(retries):
#         consumer = Consumer(_consumer_conf(config, group_id=group_id))
#         try:
#             consumer.subscribe(topics)
#             consumer.poll(timeout=join_timeout_ms / 1000.0)
#             consumer.commit(offsets=commit_tps, asynchronous=False)
#             return
#         except Exception as exc:
#             last_exc = exc
#             if "UnknownMemberIdError" not in str(exc):
#                 raise
#         finally:
#             consumer.close()

#     if last_exc is not None:
#         raise last_exc


# def build_config(
#     bootstrap_server: str,
#     security_protocol: Optional[str] = None,
#     sasl_mechanism: Optional[str] = None,
#     sasl_plain_username: Optional[str] = None,
#     sasl_plain_password: Optional[str] = None,
#     ssl_cafile: Optional[str] = None,
#     request_timeout_ms: int = 30000,
#     extra_client_kwargs: Optional[Dict[str, Any]] = None,
# ) -> KafkaNotebookConfig:
#     """
#     构建 Kafka 配置，供 notebook 里复用。

#     注意：
#     - extra_client_kwargs 必须使用 librdkafka 风格 key（点号），例如：
#         {"security.protocol": "SASL_SSL", "sasl.mechanism": "PLAIN"}
#     - request_timeout_ms 映射为 socket.timeout.ms。
#     """
#     kwargs: Dict[str, Any] = {
#         "socket.timeout.ms": request_timeout_ms,
#     }

#     if security_protocol:
#         kwargs["security.protocol"] = security_protocol
#     if sasl_mechanism:
#         kwargs["sasl.mechanism"] = sasl_mechanism
#     if sasl_plain_username:
#         kwargs["sasl.username"] = sasl_plain_username
#     if sasl_plain_password:
#         kwargs["sasl.password"] = sasl_plain_password
#     if ssl_cafile:
#         kwargs["ssl.ca.location"] = ssl_cafile

#     if extra_client_kwargs:
#         kwargs.update(extra_client_kwargs)

#     return KafkaNotebookConfig(
#         bootstrap_server=bootstrap_server,
#         client_kwargs=kwargs,
#     )


# def list_consumer_groups(config: KafkaNotebookConfig) -> List[Dict[str, str]]:
#     """查看所有消费者组。

#     注意：confluent-kafka 不再返回 protocol_type，改为返回 state。
#     """
#     admin = AdminClient(_admin_conf(config))
#     result = admin.list_consumer_groups().result()
#     return [
#         {
#             "group_id": g.group_id,
#             "state": g.state.name if getattr(g, "state", None) is not None else "",
#         }
#         for g in result.valid
#     ]


# def describe_consumer_group(
#     config: KafkaNotebookConfig,
#     group_id: str,
# ) -> List[Dict[str, int]]:
#     """查看某个消费者组消费情况，返回 committed/end/lag。"""
#     admin = AdminClient(_admin_conf(config))
#     # 用一个独立的临时 group.id 仅做 watermark 查询，避免影响目标组。
#     consumer = Consumer(_consumer_conf(config, group_id=f"__notebook_describe_{group_id}"))

#     try:
#         committed_tps = _list_group_offsets(admin, group_id)
#         if not committed_tps:
#             return []

#         committed_tps = sorted(committed_tps, key=lambda tp: (tp.topic, tp.partition))

#         rows: List[Dict[str, int]] = []
#         for tp in committed_tps:
#             committed_offset = tp.offset if tp.offset is not None else -1
#             try:
#                 _low, log_end_offset = consumer.get_watermark_offsets(
#                     TopicPartition(tp.topic, tp.partition), timeout=10.0
#                 )
#             except KafkaException:
#                 log_end_offset = -1
#             lag = (
#                 log_end_offset - committed_offset
#                 if committed_offset >= 0 and log_end_offset >= 0
#                 else -1
#             )
#             rows.append(
#                 {
#                     "topic": tp.topic,
#                     "partition": tp.partition,
#                     "committed_offset": committed_offset,
#                     "log_end_offset": log_end_offset,
#                     "lag": lag,
#                 }
#             )
#         return rows
#     finally:
#         consumer.close()


# def set_consumer_group_offsets(
#     config: KafkaNotebookConfig,
#     group_id: str,
#     topic: str,
#     partition_offsets: Dict[int, int],
#     execute: bool = False,
# ) -> Dict[str, Any]:
#     """
#     修改消费者组 offset。

#     参数：
#     - partition_offsets 示例：{0: 100, 1: 200}
#     - execute=False 时仅 dry-run

#     实现策略：
#     - 首选 Admin API 直接修改消费组 offset（不依赖 member 身份，稳定性更好）。
#     - 若环境不支持或触发 UnknownMemberIdError，则回退到 consumer commit 兜底。
#     """
#     admin = AdminClient(_admin_conf(config))

#     target_tps = [
#         TopicPartition(topic, p) for p in sorted(partition_offsets.keys())
#     ]

#     existing_list = _list_group_offsets(admin, group_id)
#     existing_map = {(tp.topic, tp.partition): tp for tp in existing_list}

#     plan = []
#     for tp in target_tps:
#         current = existing_map.get((tp.topic, tp.partition))
#         current_offset = current.offset if current is not None else -1
#         plan.append(
#             {
#                 "topic": tp.topic,
#                 "partition": tp.partition,
#                 "current_offset": current_offset,
#                 "target_offset": partition_offsets[tp.partition],
#             }
#         )

#     if not execute:
#         return {
#             "mode": "dry-run",
#             "plan": plan,
#         }

#     commit_tps = [
#         TopicPartition(
#             topic=tp.topic,
#             partition=tp.partition,
#             offset=partition_offsets[tp.partition],
#             metadata="set-by-kafka-group-operator-notebook",
#         )
#         for tp in target_tps
#     ]

#     applied_strategy = "admin.alter_consumer_group_offsets"

#     try:
#         futures = admin.alter_consumer_group_offsets(
#             [ConsumerGroupTopicPartitions(group_id, commit_tps)]
#         )
#         futures[group_id].result()
#     except Exception as exc:
#         if "UnknownMemberIdError" not in str(exc):
#             raise
#         _commit_offsets_via_consumer_with_retry(
#             config=config,
#             group_id=group_id,
#             topics=[topic],
#             commit_tps=commit_tps,
#         )
#         applied_strategy = "consumer.commit(retry-after-join)"

#     latest_list = _list_group_offsets(admin, group_id)
#     latest_map = {(tp.topic, tp.partition): tp for tp in latest_list}
#     after = []
#     for tp in target_tps:
#         latest = latest_map.get((tp.topic, tp.partition))
#         after.append(
#             {
#                 "topic": tp.topic,
#                 "partition": tp.partition,
#                 "offset_after_commit": latest.offset if latest is not None else -1,
#             }
#         )

#     return {
#         "mode": "execute",
#         "strategy": applied_strategy,
#         "plan": plan,
#         "after": after,
#     }


# def set_group_offsets_to_timestamp_prev(
#     config: KafkaNotebookConfig,
#     group_id: str,
#     timestamp_ms: Any,
#     execute: bool = False,
# ) -> Dict[str, Any]:
#     """
#     将指定消费者组下所有已提交分区的 offset 调整到“指定时间戳之前一条”。

#         说明：
#         - 作用范围：该消费组当前已提交过 offset 的全部 topic/partition。
#         - 目标值计算：
#             1) 若存在 timestamp_ms 对应位置 offset_x，则目标为 max(offset_x - 1, 0)
#             2) 若不存在（通常是时间戳晚于分区最后一条消息时间），目标为 max(end_offset - 1, 0)
#         - execute=False 时仅返回计划，不会真正提交。

#         设计意图：
#         - 该方法面向“按时间点整体回拨”场景，避免逐 topic/partition 手工计算 offset。
#         - 返回 plan/after 两段数据，便于先预演、再执行、最后核对。
#     """
#     parsed_timestamp_ms = parse_time_to_timestamp_ms(timestamp_ms)

#     admin = AdminClient(_admin_conf(config))
#     consumer = Consumer(
#         _consumer_conf(config, group_id=f"__notebook_ts_{group_id}")
#     )

#     try:
#         existing_list = _list_group_offsets(admin, group_id)
#         if not existing_list:
#             return {
#                 "mode": "dry-run" if not execute else "execute",
#                 "group_id": group_id,
#                 "timestamp_ms": parsed_timestamp_ms,
#                 "plan": [],
#                 "message": "该消费者组没有已提交 offset，未执行任何修改。",
#             }

#         target_tps = sorted(existing_list, key=lambda tp: (tp.topic, tp.partition))

#         # 在 confluent-kafka 中，通过将 TopicPartition 的 offset 字段设为时间戳传给 offsets_for_times。
#         ts_query = [
#             TopicPartition(tp.topic, tp.partition, parsed_timestamp_ms)
#             for tp in target_tps
#         ]
#         ts_results = consumer.offsets_for_times(ts_query, timeout=10.0)
#         ts_map = {(tp.topic, tp.partition): tp for tp in ts_results}

#         end_offsets_map: Dict[tuple, int] = {}
#         for tp in target_tps:
#             try:
#                 _low, high = consumer.get_watermark_offsets(
#                     TopicPartition(tp.topic, tp.partition), timeout=10.0
#                 )
#                 end_offsets_map[(tp.topic, tp.partition)] = high
#             except KafkaException:
#                 end_offsets_map[(tp.topic, tp.partition)] = 0

#         plan = []
#         commit_tps: List[TopicPartition] = []
#         for tp in target_tps:
#             current_offset = tp.offset if tp.offset is not None else -1

#             ts_tp = ts_map.get((tp.topic, tp.partition))
#             end_offset = end_offsets_map.get((tp.topic, tp.partition), 0)

#             # 在 confluent-kafka 中，offsets_for_times 找不到时返回 offset = -1。
#             if ts_tp is not None and ts_tp.offset is not None and ts_tp.offset >= 0:
#                 target_offset = max(ts_tp.offset - 1, 0)
#                 calc_reason = "from_offsets_for_times_minus_1"
#             else:
#                 target_offset = max(end_offset - 1, 0)
#                 calc_reason = "timestamp_after_last_or_empty_partition"

#             commit_tps.append(
#                 TopicPartition(
#                     topic=tp.topic,
#                     partition=tp.partition,
#                     offset=target_offset,
#                     metadata="set-by-group-timestamp-prev",
#                 )
#             )
#             plan.append(
#                 {
#                     "topic": tp.topic,
#                     "partition": tp.partition,
#                     "current_offset": current_offset,
#                     "target_offset": target_offset,
#                     "end_offset": end_offset,
#                     "calc_reason": calc_reason,
#                 }
#             )

#         if not execute:
#             return {
#                 "mode": "dry-run",
#                 "group_id": group_id,
#                 "timestamp_ms": parsed_timestamp_ms,
#                 "plan": plan,
#             }

#         applied_strategy = "admin.alter_consumer_group_offsets"
#         try:
#             futures = admin.alter_consumer_group_offsets(
#                 [ConsumerGroupTopicPartitions(group_id, commit_tps)]
#             )
#             futures[group_id].result()
#         except Exception as exc:
#             if "UnknownMemberIdError" not in str(exc):
#                 raise
#             _commit_offsets_via_consumer_with_retry(
#                 config=config,
#                 group_id=group_id,
#                 topics=sorted(list({tp.topic for tp in target_tps})),
#                 commit_tps=commit_tps,
#             )
#             applied_strategy = "consumer.commit(retry-after-join)"

#         latest_list = _list_group_offsets(admin, group_id)
#         latest_map = {(tp.topic, tp.partition): tp for tp in latest_list}
#         after = []
#         for tp in target_tps:
#             latest = latest_map.get((tp.topic, tp.partition))
#             after.append(
#                 {
#                     "topic": tp.topic,
#                     "partition": tp.partition,
#                     "offset_after_commit": latest.offset if latest is not None else -1,
#                 }
#             )

#         return {
#             "mode": "execute",
#             "group_id": group_id,
#             "timestamp_ms": parsed_timestamp_ms,
#             "strategy": applied_strategy,
#             "plan": plan,
#             "after": after,
#         }
#     finally:
#         consumer.close()


# def print_rows(rows: List[Dict[str, Any]]) -> None:
#     """轻量表格输出，方便 notebook 快速查看。"""
#     if not rows:
#         print("(empty)")
#         return

#     headers = list(rows[0].keys())
#     col_widths = {h: len(h) for h in headers}

#     for row in rows:
#         for h in headers:
#             col_widths[h] = max(col_widths[h], len(str(row.get(h, ""))))

#     header_line = " | ".join(h.ljust(col_widths[h]) for h in headers)
#     sep_line = "-+-".join("-" * col_widths[h] for h in headers)
#     print(header_line)
#     print(sep_line)

#     for row in rows:
#         print(" | ".join(str(row.get(h, "")).ljust(col_widths[h]) for h in headers))

# def to_json(value: str) -> str:
#     import json
#     return json.dumps(result)

In [0]:
bootstrap_server = "10.249.200.43:9092,10.249.200.42:9092,10.249.200.44:9092"

topic_list = "ConsumerTopic,Consumer_NZ,Consumer_HK,Consumer_ID,Consumer_JP,Consumer_KR,Consumer_MY,Consumer_PH,Consumer_SG,Consumer_TH,Consumer_TW,Consumer_VN,Consumer_JP_Rakuten,Consumer_TW_Linegift"

group_id = "cdp-mdm-validate-consumer-prod"



def read_kafka_for_consumergroup(bootstrap_server, group_id, topics):
    topics = [t.strip() for t in topic_list.split(",")]

    consumer = KafkaConsumer(
        *topics,
        bootstrap_servers=bootstrap_server,
        group_id=group_id,
        auto_offset_reset="latest",       # 仅拉取新到达的消息
        enable_auto_commit=True,
        value_deserializer=lambda v: v.decode("utf-8"),
        key_deserializer=lambda k: k.decode("utf-8") if k else None
    )

    try:
        for message in consumer:
            print(
                f"topic={message.topic} | partition={message.partition} | "
                f"offset={message.offset} | key={message.key} " #| value={message.value}
            )
    except KeyboardInterrupt:
        pass
    finally:
        consumer.close()

read_kafka_for_consumergroup(bootstrap_server, group_id, topic_list)

In [0]:
display(describe_consumer_group(cfg, group_id))

In [0]:
cfg = build_config(bootstrap_server=bootstrap_server)

# 1) 查看所有消费者组
groups = list_consumer_groups(cfg)
# display(groups)

# 2) 查看某个组消费情况
status_rows = describe_consumer_group(cfg, group_id=group_id)
display(status_rows)


# 3) 修改 offset (先 dry-run)
# plan = set_consumer_group_offsets(
#   cfg,
#   group_id=group_id,
#   topic="Consumer_HK_TS",
#   partition_offsets={0: 248076},
#   execute=False,
# )
# print(plan)



In [0]:
# 4) 修改 offset (真正执行)
# result = set_consumer_group_offsets(
#   cfg,
#   group_id=group_id,
#   topic="Consumer_HK_TS",
#   partition_offsets={0: 248076},
#   execute=True,
# )
# print(result)

# 5) 修改offset - 基于时间戳/时间字符串 (支持dry-run / 真正执行)

# "2026-05-27T08:00:00Z"
timestamp_ms = "2026-07-15T00:00:00Z"

result = set_group_offsets_to_timestamp_prev(
    cfg,
    group_id=group_id,
    timestamp_ms=timestamp_ms,
    # FALSE - dry-run, TRUE - 真正运行
    execute=True
)
print(to_json(result))

In [0]:
import pyspark.sql.functions as F

kafka_df = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_server) \
    .option("subscribe", "ConsumerTopic") \
    .option("kafka.group.id", group_id) \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()


display(kafka_df.where("partition = 0 and offset = 450097"))

In [0]:
display(
    spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_server) \
    .option("subscribe", "ConsumerTopic_VL,Consumer_NZ_VL,Consumer_HK_VL,Consumer_ID_VL,Consumer_JP_VL,Consumer_KR_VL,Consumer_MY_VL,Consumer_PH_VL,Consumer_SG_VL,Consumer_TH_VL,Consumer_TW_VL,Consumer_VN_VL,Consumer_JP_Rakuten_VL,Consumer_TW_Linegift_VL,Consumer_History_VL") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()
    .groupBy("topic")
    .agg(
        F.count("*"),
        F.min("timestamp"),
        F.max("timestamp")
    )

)

In [0]:
%sql
-- insert into 
--     catalog_southeastasia_mdm_pr.share_mdm_config.t_kafka_last_read(task_id, batch_id, topic, last_end_ms, last_end_dt, read_record_count, creation_time)
-- values
-- ('init_TouchPointTopic', 'init_TouchPointTopic', 'TouchPointTopic', 1781020800000, '2026-06-09 16:00:00', 0, current_timestamp()),
-- ('init_TouchPoint_KR', 'init_TouchPoint_KR', 'TouchPoint_KR', 1781020800000, '2026-06-09 16:00:00', 0, current_timestamp())

In [0]:
%sql

select
    *
from
    catalog_southeastasia_mdm_pr.share_mdm_config.t_kafka_last_read